# IAN 603 · Module 7
# Subqueries in PostgreSQL

A subquery is a query used inside another query.

The most useful way to choose a subquery pattern is not by memorizing syntax. Ask:

> **What shape should the inner query return?**

Use this decision process:

```text
Need one value?
    → scalar subquery

Need a set of values?
    → IN / multi-row comparison

Need to know whether a row exists?
    → EXISTS / NOT EXISTS

Need a result that depends on the current outer row?
    → correlated subquery

Need a temporary table-like result?
    → derived table or CTE
```

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module7.html


## What you will learn

By the end of this notebook, you should be able to:

1. Identify the expected result shape of a subquery.
2. Use scalar subqueries in `WHERE` and `SELECT`.
3. Explain what happens when a scalar subquery returns zero rows.
4. Recognize the error caused by a scalar subquery returning multiple rows.
5. Use `IN` for membership tests.
6. Explain why `IN` and `JOIN` can produce different row counts.
7. Explain the `NOT IN + NULL` trap.
8. Use `EXISTS` and `NOT EXISTS`.
9. Use `ANY` / `SOME` and `ALL`.
10. Explain empty-set behavior for `ANY` and `ALL`.
11. Recognize and trace a correlated subquery.
12. Use correlated counts and correlated benchmarks.
13. Use a derived table in `FROM`.
14. Rewrite some correlated queries as pre-aggregation + join.
15. Use `EXPLAIN` to inspect a query plan.
16. Validate inner-query shape and outer-query grain.


## 0. Install required Python packages

Run this cell if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** The helper is downloaded automatically when needed.

If you want your own copy:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)


In [ ]:
from pathlib import Path
import urllib.request
from sqlalchemy import text

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password and click **Connect**.


In [ ]:
db.show_connection_panel()


### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 3. Check the connection

After clicking **Connect**, run this cell once.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Meet the Teaching Database

The examples use related patient, admission, lab, prescription, and review tables.

The important grains are:

| Table | One row represents |
|---|---|
| `patients` | one patient |
| `admissions` | one admission |
| `lab_results` | one lab test |
| `prescriptions` | one prescription |
| `review_queue` | one review-queue record |

Subqueries often compare one grain with another.

For example:

- patient row compared with overall average age,
- patient row checked against a set of admitted patient IDs,
- lab row compared with average for the same test,
- patient row checked for existence of an abnormal lab.


## 4. Create or reset the Module 7 teaching database


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 7\n-- Subqueries in PostgreSQL\n-- ============================================================\n-- Companion file for s_module7.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- scalar and multi-row subqueries, IN, EXISTS, NOT EXISTS,\n-- correlated subqueries, derived tables, nested logic, and CTEs.\n--\n-- Choose a pattern based on the shape of the value or row set\n-- needed by the outer query.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS review_queue CASCADE;\nDROP TABLE IF EXISTS prescriptions CASCADE;\nDROP TABLE IF EXISTS lab_results CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS departments CASCADE;\nDROP TABLE IF EXISTS patients CASCADE;\n\nCREATE TABLE patients (\n    patient_id integer PRIMARY KEY,\n    name varchar(100) NOT NULL,\n    age integer NOT NULL CHECK (age BETWEEN 0 AND 120),\n    gender varchar(20),\n    home_city varchar(60) NOT NULL\n);\n\nCREATE TABLE departments (\n    department varchar(60) PRIMARY KEY,\n    service_line varchar(80) NOT NULL\n);\n\nCREATE TABLE admissions (\n    admission_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    admit_date date NOT NULL,\n    discharge_date date,\n    diagnosis varchar(120) NOT NULL,\n    department varchar(60) NOT NULL REFERENCES departments(department),\n    CHECK (discharge_date IS NULL OR discharge_date >= admit_date)\n);\n\nCREATE TABLE lab_results (\n    test_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    test_name varchar(60) NOT NULL,\n    result_value numeric(10,2),\n    test_date date NOT NULL,\n    normal_low numeric(10,2),\n    normal_high numeric(10,2)\n);\n\nCREATE TABLE prescriptions (\n    rx_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patients(patient_id),\n    medication varchar(100) NOT NULL,\n    dosage varchar(50) NOT NULL,\n    prescribed_date date NOT NULL,\n    duration_days integer CHECK (duration_days > 0)\n);\n\n-- The NULL is deliberate: it lets students see why NOT IN can be unsafe.\nCREATE TABLE review_queue (\n    queue_id integer PRIMARY KEY,\n    patient_id integer\n);\n\nINSERT INTO patients VALUES\n(1001,'Alice Smith',45,'Female','Greensboro'),\n(1002,'Bob Johnson',32,'Male','High Point'),\n(1003,'Carol White',58,'Female','Greensboro'),\n(1004,'David Brown',67,'Male','Burlington'),\n(1005,'Eve Davis',29,'Female','Greensboro'),\n(1006,'Frank Miller',71,'Male','Reidsville'),\n(1007,'Grace Wilson',52,'Female','Greensboro'),\n(1008,'Henry Taylor',48,'Male','High Point'),\n(1009,'Ivy Clark',63,'Female','Greensboro'),\n(1010,'Jack Anderson',55,'Male','Burlington'),\n(1011,'Kara Young',41,'Female','Greensboro');\n\nINSERT INTO departments VALUES\n('Cardiology','Cardiovascular'),\n('Endocrinology','Medicine'),\n('Internal Medicine','Medicine'),\n('Neurology','Neurosciences'),\n('Orthopedics','Musculoskeletal'),\n('Pulmonology','Medicine');\n\nINSERT INTO admissions VALUES\n(101,1001,'2026-01-15','2026-01-20','Pneumonia','Internal Medicine'),\n(102,1002,'2026-01-16','2026-01-18','Fractured arm','Orthopedics'),\n(103,1003,'2026-01-17','2026-01-25','Cardiac arrhythmia','Cardiology'),\n(104,1001,'2026-03-05','2026-03-12','Hypertension','Internal Medicine'),\n(105,1004,'2026-02-19','2026-02-26','Heart disease','Cardiology'),\n(106,1005,'2026-04-20','2026-04-22','Migraine','Neurology'),\n(107,1006,'2026-05-21','2026-05-28','Diabetes complications','Endocrinology'),\n(108,1007,'2026-05-22','2026-05-24','Asthma','Pulmonology'),\n(109,1008,'2026-06-23','2026-06-30','Cardiac monitoring','Cardiology'),\n(110,1009,'2026-07-24','2026-07-27','Joint pain','Orthopedics'),\n(111,1010,'2026-07-25',NULL,'Hypertension','Internal Medicine'),\n(112,1006,'2026-08-01','2026-08-05','Cardiac assessment','Cardiology');\n\nINSERT INTO lab_results VALUES\n(201,1001,'Glucose',98,'2026-01-16',70,110),\n(202,1002,'Cholesterol',185,'2026-01-17',0,200),\n(203,1003,'Glucose',145,'2026-01-18',70,110),\n(204,1004,'Systolic BP',150,'2026-02-19',90,140),\n(205,1005,'Cholesterol',210,'2026-04-20',0,200),\n(206,1006,'Glucose',320,'2026-05-21',70,110),\n(207,1007,'Systolic BP',128,'2026-05-22',90,140),\n(208,1008,'Cholesterol',195,'2026-06-23',0,200),\n(209,1009,'Glucose',110,'2026-07-24',70,110),\n(210,1010,'Systolic BP',142,'2026-07-25',90,140),\n(211,1001,'Hemoglobin',14.2,'2026-01-16',12,16),\n(212,1003,'Hemoglobin',11.8,'2026-01-18',12,16),\n(213,1006,'Hemoglobin',10.9,'2026-05-21',12,16);\n\nINSERT INTO prescriptions VALUES\n(301,1001,'Amoxicillin','500mg','2026-01-16',7),\n(302,1002,'Ibuprofen','400mg','2026-01-17',14),\n(303,1003,'Metoprolol','25mg','2026-01-18',90),\n(304,1004,'Atorvastatin','40mg','2026-02-19',30),\n(305,1006,'Metformin','1000mg','2026-05-21',60),\n(306,1006,'Insulin','50 units','2026-05-21',90),\n(307,1001,'Lisinopril','5mg','2026-03-06',30),\n(308,1007,'Albuterol','2 puffs','2026-05-22',15),\n(309,1008,'Aspirin','81mg','2026-06-23',365),\n(310,1009,'Ibuprofen','400mg','2026-07-24',7);\n\nINSERT INTO review_queue VALUES\n(1,1003),\n(2,NULL);"

execute_script(setup_sql)


## 5. Preview the patients first


In [ ]:
display_query("""
SELECT patient_id, name, age, home_city
FROM patients
ORDER BY patient_id;
""", title="Patients")


# Part II · Shape Before Syntax

Before writing the outer query, ask what the inner query should return.

### Shape 1 — one value

```text
42.6
```

Use a **scalar subquery**.

### Shape 2 — several values

```text
1001
1003
1006
```

Use `IN`, `ANY`, `ALL`, or another multi-row pattern.

### Shape 3 — existence only

```text
Does at least one matching row exist?
```

Use `EXISTS` or `NOT EXISTS`.

### Shape 4 — table-like result

```text
patient_id | admission_count
```

Use a derived table or CTE.

Choosing the shape first makes the syntax much easier.


# Part III · Scalar Subqueries

A scalar subquery must return:

- exactly one column,
- at most one row.

It behaves like a single value.


## 6. Run the inner query first

Question:

> Which patients are older than the overall average age?

First calculate the benchmark by itself.


In [ ]:
display_query("""
SELECT ROUND(AVG(age), 2) AS average_age
FROM patients;
""", title="Inner query: one scalar value")


The inner result shape is:

| average_age |
|---:|
| one value |

That makes it safe to place where SQL expects one value.


## 7. Embed the scalar in WHERE


In [ ]:
display_query("""
SELECT
    patient_id,
    name,
    age
FROM patients
WHERE age > (
    SELECT AVG(age)
    FROM patients
)
ORDER BY age DESC;
""", title="Patients older than average")


### Read it from inside out

1. Inner query calculates one average age.
2. Outer query examines each patient.
3. Patient age is compared with that one benchmark.
4. Only TRUE rows remain.


## 8. A scalar subquery can also appear in SELECT

The inner query is evaluated as a value and repeated beside each outer row.


In [ ]:
display_query("""
SELECT
    patient_id,
    name,
    age,
    (
        SELECT ROUND(AVG(age), 1)
        FROM patients
    ) AS hospital_avg_age
FROM patients
ORDER BY patient_id;
""", title="Overall average beside each patient")


### Grain

The scalar subquery does not change the outer grain.

The result is still:

> **one row per patient**

It simply adds a benchmark column.


# Part IV · Scalar Edge Cases

Scalar subqueries have two important edge cases.

### Zero inner rows

A scalar subquery returning zero rows becomes `NULL`.

### More than one inner row

That is an error because SQL expected one value but received several.


## 9. Zero matching rows → NULL


In [ ]:
display_query("""
SELECT (
    SELECT result_value
    FROM lab_results
    WHERE test_id = 9999
) AS missing_scalar_result;
""", title="Scalar subquery with zero rows")


## 10. Too many rows → controlled error

This query is intentionally invalid.


In [ ]:
try:
    with db.engine.connect() as conn:
        conn.execute(text("""
            SELECT (
                SELECT patient_id
                FROM admissions
            ) AS patient_id;
        """))
except Exception as e:
    print("✅ Expected scalar-subquery error")
    print()
    print(str(e).split("\n")[0])


### Scalar checklist

Before embedding a scalar subquery, ask:

1. Does it return one column?
2. Can it return more than one row?
3. What should happen if it returns zero rows?
4. Does NULL need special handling?


# Part V · IN Means Membership

Use `IN` when the inner query returns a set of candidate values.

Question:

> Which patients have at least one Cardiology admission?

The inner query should return:

> a set of patient IDs from Cardiology admissions.


## 11. Run the membership set by itself


In [ ]:
display_query("""
SELECT DISTINCT patient_id
FROM admissions
WHERE department = 'Cardiology'
ORDER BY patient_id;
""", title="Cardiology patient ID set")


## 12. Use that set with IN


In [ ]:
display_query("""
SELECT
    patient_id,
    name,
    age
FROM patients
WHERE patient_id IN (
    SELECT patient_id
    FROM admissions
    WHERE department = 'Cardiology'
)
ORDER BY patient_id;
""", title="Patients in the Cardiology set")


# Part VI · IN Is Not the Same as JOIN

These two questions are different.

### Membership question

> Is this patient ID present in the admissions set?

`IN` returns each outer patient row once.

### Combination question

> Show each patient-admission combination.

A `JOIN` can repeat a patient when several admissions match.


## 13. Compare row shape directly


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name
FROM patients AS p
WHERE p.patient_id IN (
    SELECT a.patient_id
    FROM admissions AS a
)
ORDER BY p.patient_id;
""", title="IN: one row per matching patient")


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    a.admission_id
FROM patients AS p
JOIN admissions AS a
    ON a.patient_id = p.patient_id
ORDER BY p.patient_id, a.admission_id;
""", title="JOIN: one row per patient-admission match")


The right choice depends on the question and desired grain.

Do not replace `IN` with a join automatically.


# Part VII · The NOT IN + NULL Trap

`NOT IN` looks natural for anti-matching, but it becomes dangerous when the inner set contains `NULL`.

The teaching table `review_queue` intentionally contains:

- one real patient ID,
- one NULL patient ID.

That NULL can make many comparisons evaluate to UNKNOWN.


## 14. Inspect the review queue first


In [ ]:
display_query("""
SELECT queue_id, patient_id, reason
FROM review_queue
ORDER BY queue_id;
""", title="review_queue contains a NULL patient_id")


## 15. See NOT IN behave unexpectedly


In [ ]:
display_query("""
SELECT
    patient_id,
    name
FROM patients
WHERE patient_id NOT IN (
    SELECT patient_id
    FROM review_queue
)
ORDER BY patient_id;
""", title="NOT IN with NULL in the inner set")


### Why?

Conceptually, SQL must consider comparisons such as:

```text
1001 <> 1003 AND 1001 <> NULL
```

The second comparison is UNKNOWN.

TRUE AND UNKNOWN becomes UNKNOWN.

`WHERE` keeps only TRUE, so the row disappears.


# Part VIII · NOT EXISTS: Safer Anti-Match

For the question:

> Keep patients for whom no matching review_queue row exists,

`NOT EXISTS` expresses the logic directly.


## 16. Rewrite the anti-match with NOT EXISTS


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name
FROM patients AS p
WHERE NOT EXISTS (
    SELECT 1
    FROM review_queue AS q
    WHERE q.patient_id = p.patient_id
)
ORDER BY p.patient_id;
""", title="NOT EXISTS anti-match")


### How to read EXISTS

`SELECT 1` does not mean the value 1 is important.

It means:

> I only care whether at least one matching row exists.

The database may optimize this as an existence test.


# Part IX · EXISTS

Use `EXISTS` when the question is about presence, not about returning inner values.

Question:

> Which patients have at least one abnormal lab?


## 17. Define abnormality inside the correlated inner query

A lab is abnormal when:

```text
result < normal_low
OR
result > normal_high
```

The inner query also connects to the current patient:

```sql
l.patient_id = p.patient_id
```


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    p.age
FROM patients AS p
WHERE EXISTS (
    SELECT 1
    FROM lab_results AS l
    WHERE l.patient_id = p.patient_id
      AND (
          l.result_value < l.normal_low
          OR l.result_value > l.normal_high
      )
)
ORDER BY p.age DESC;
""", title="Patients with at least one abnormal lab")


## 18. NOT EXISTS: patients with no prescriptions


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name
FROM patients AS p
WHERE NOT EXISTS (
    SELECT 1
    FROM prescriptions AS pr
    WHERE pr.patient_id = p.patient_id
)
ORDER BY p.patient_id;
""", title="Patients with no prescriptions")


# Part X · IN vs EXISTS

Both can express related ideas, but their mental models differ.

| Pattern | Think |
|---|---|
| `IN` | Is this value a member of this set? |
| `EXISTS` | Does at least one matching row exist? |

For simple membership, `IN` is often very readable.

For correlated existence logic, especially with multiple inner conditions, `EXISTS` is often clearer.


# Part XI · ANY / SOME and ALL

These operators compare one value with a set of values.

### ANY / SOME

Condition is TRUE if the comparison succeeds for **at least one** inner value.

### ALL

Condition is TRUE only if the comparison succeeds for **every** inner value.


## 19. First inspect the Cardiology ages

Run the inner set before comparing anything.


In [ ]:
display_query("""
SELECT DISTINCT p.age
FROM patients AS p
JOIN admissions AS a
    ON a.patient_id = p.patient_id
WHERE a.department = 'Cardiology'
ORDER BY p.age;
""", title="Ages in the Cardiology set")


## 20. age > ANY (...)

Interpretation:

> Patient age is greater than at least one Cardiology age.

With numeric values, this is often similar to being greater than the minimum value in the set.


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    p.age
FROM patients AS p
WHERE p.age > ANY (
    SELECT p2.age
    FROM patients AS p2
    JOIN admissions AS a
        ON a.patient_id = p2.patient_id
    WHERE a.department = 'Cardiology'
)
ORDER BY p.age DESC;
""", title="age > ANY Cardiology age")


## 21. age > ALL (...)

Interpretation:

> Patient age is greater than every Cardiology age.

For numeric values, this is similar to being greater than the maximum value in the set.


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    p.age
FROM patients AS p
WHERE p.age > ALL (
    SELECT p2.age
    FROM patients AS p2
    JOIN admissions AS a
        ON a.patient_id = p2.patient_id
    WHERE a.department = 'Cardiology'
)
ORDER BY p.age DESC;
""", title="age > ALL Cardiology ages")


# Part XII · Empty-Set Behavior

Empty sets produce a useful logical result.

```text
x > ANY(empty set)  → FALSE
x > ALL(empty set)  → TRUE
```

Why?

- `ANY`: there is no value that satisfies the comparison.
- `ALL`: there is no value that violates the comparison.

This is called vacuous truth for `ALL`.


## 22. Test ANY and ALL on an empty set


In [ ]:
display_query("""
SELECT
    45 > ANY (
        SELECT age
        FROM patients
        WHERE patient_id < 0
    ) AS greater_than_any_empty,
    45 > ALL (
        SELECT age
        FROM patients
        WHERE patient_id < 0
    ) AS greater_than_all_empty;
""", title="ANY and ALL with an empty set")


# Part XIII · Correlated Subqueries

A correlated subquery references a column from the outer query.

That means the inner query changes depending on the current outer row.

Look for an inner condition like:

```sql
l2.test_name = l.test_name
```

The inner query is using `l.test_name` from the current outer lab row.


## 23. Trace one correlated benchmark manually

Suppose the current outer row is:

| test_id | test_name | result |
|---:|---|---:|
| 501 | Glucose | 140 |

The inner query becomes conceptually:

```sql
SELECT AVG(result_value)
FROM lab_results
WHERE test_name = 'Glucose';
```

For the next outer row, the test name may change, so the benchmark changes too.


## 24. Compare each lab with the average for the same test


In [ ]:
display_query("""
SELECT
    l.test_id,
    l.patient_id,
    l.test_name,
    l.result_value,
    ROUND((
        SELECT AVG(l2.result_value)
        FROM lab_results AS l2
        WHERE l2.test_name = l.test_name
    ), 2) AS test_average
FROM lab_results AS l
WHERE l.result_value > (
    SELECT AVG(l3.result_value)
    FROM lab_results AS l3
    WHERE l3.test_name = l.test_name
)
ORDER BY l.test_name, l.result_value DESC;
""", title="Above the average for the same test")


### Outer grain

The result remains:

> **one row per qualifying lab result**

The correlated subquery supplies a row-specific benchmark; it does not automatically change the outer grain.


# Part XIV · Correlated Subquery in SELECT

A correlated subquery can calculate one value for every outer row.

Question:

> How many admissions does each patient have?


## 25. Admission count beside each patient


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    (
        SELECT COUNT(*)
        FROM admissions AS a
        WHERE a.patient_id = p.patient_id
    ) AS admission_count
FROM patients AS p
ORDER BY admission_count DESC, p.patient_id;
""", title="Correlated admission count")


### Why zero appears naturally

`COUNT(*)` over zero matching inner rows returns 0.

So patients with no admissions still receive a numeric count.


# Part XV · Latest Related Row

A correlated scalar subquery can also define a row-specific maximum.

Question:

> For each patient, keep the admission whose date equals that patient's latest admission date.


## 26. Latest admission per patient


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    a.admission_id,
    a.admit_date,
    a.diagnosis
FROM patients AS p
JOIN admissions AS a
    ON a.patient_id = p.patient_id
WHERE a.admit_date = (
    SELECT MAX(a2.admit_date)
    FROM admissions AS a2
    WHERE a2.patient_id = p.patient_id
)
ORDER BY p.patient_id;
""", title="Latest admission per patient")


### Tie warning

If two admissions for the same patient have exactly the same maximum date, this query can return both rows.

If the requirement is **exactly one row per patient**, define a tie-breaking rule.


# Part XVI · Subquery Placement

A subquery can appear in several places.

### WHERE

Use it to filter.

### SELECT

Use it to calculate a value beside each outer row.

### FROM

Use it as a temporary table-like result.

The placement often reveals the role of the subquery.


# Part XVII · Derived Tables

A derived table is a subquery inside `FROM`.

It must have an alias.

Example goal:

> First count admissions per patient, then join those counts to all patients.


## 27. Build the derived table by itself first


In [ ]:
display_query("""
SELECT
    patient_id,
    COUNT(*) AS admission_count
FROM admissions
GROUP BY patient_id
ORDER BY patient_id;
""", title="Derived-table content")


## 28. Join the derived table to patients


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    COALESCE(ac.admission_count, 0) AS admission_count
FROM patients AS p
LEFT JOIN (
    SELECT
        patient_id,
        COUNT(*) AS admission_count
    FROM admissions
    GROUP BY patient_id
) AS ac
    ON ac.patient_id = p.patient_id
ORDER BY admission_count DESC, p.patient_id;
""", title="Admission counts from a derived table")


### Grain reasoning

Inner derived table:

> one row per patient with admissions

Outer result:

> one row per patient

The `LEFT JOIN` preserves patients with no admissions.


# Part XVIII · Correlated vs Pre-Aggregated Design

These two approaches can answer the same question.

### Correlated count

```text
for each patient
    → count that patient's admissions
```

### Pre-aggregate + join

```text
count admissions by patient once
    ↓
join summary to patients
```

For small teaching data, both are fine.

For larger analytical workflows, pre-aggregation can be easier to reason about and may be more efficient.


## 29. Compare the two styles


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    (
        SELECT COUNT(*)
        FROM admissions AS a
        WHERE a.patient_id = p.patient_id
    ) AS admission_count
FROM patients AS p
ORDER BY p.patient_id;
""", title="Correlated style")


In [ ]:
display_query("""
SELECT
    p.patient_id,
    p.name,
    COALESCE(a.admission_count, 0) AS admission_count
FROM patients AS p
LEFT JOIN (
    SELECT
        patient_id,
        COUNT(*) AS admission_count
    FROM admissions
    GROUP BY patient_id
) AS a USING (patient_id)
ORDER BY p.patient_id;
""", title="Pre-aggregated style")


# Part XIX · CTEs as Named Stages

A CTE can make a derived-table idea easier to read when the intermediate result deserves a name.

The CTE below creates patient features first, then calculates a risk score.


## 30. Build a patient feature table in stages


In [ ]:
display_query("""
WITH patient_features AS (
    SELECT
        p.patient_id,
        p.name,
        p.age,
        (
            SELECT COUNT(*)
            FROM admissions AS a
            WHERE a.patient_id = p.patient_id
        ) AS admissions,
        (
            SELECT COUNT(*)
            FROM lab_results AS l
            WHERE l.patient_id = p.patient_id
              AND (
                  l.result_value < l.normal_low
                  OR l.result_value > l.normal_high
              )
        ) AS abnormal_labs,
        (
            SELECT COUNT(DISTINCT medication)
            FROM prescriptions AS pr
            WHERE pr.patient_id = p.patient_id
        ) AS medications
    FROM patients AS p
)
SELECT
    *,
    (
        CASE
            WHEN age >= 65 THEN 3
            WHEN age >= 50 THEN 2
            ELSE 1
        END
        + CASE
            WHEN admissions >= 2 THEN 2
            WHEN admissions = 1 THEN 1
            ELSE 0
        END
        + LEAST(abnormal_labs, 2)
        + LEAST(medications, 2)
    ) AS risk_score
FROM patient_features
ORDER BY risk_score DESC, patient_id;
""", title="Patient feature table and risk score")


### Why name the stage?

The feature logic and the scoring logic are different responsibilities.

Naming the feature stage makes the final query easier to inspect and test.


# Part XX · Department Dashboard Example

Subqueries can also add a specialized metric to a grouped query.

Here, the main query groups admissions by department while a correlated scalar subquery counts currently open admissions for that department.


In [ ]:
display_query("""
SELECT
    a.department,
    COUNT(*) AS admissions,
    COUNT(DISTINCT a.patient_id) AS patients,
    ROUND(
        AVG(a.discharge_date - a.admit_date),
        1
    ) AS avg_stay_days,
    (
        SELECT COUNT(*)
        FROM admissions AS x
        WHERE x.department = a.department
          AND x.discharge_date IS NULL
    ) AS current_open_admissions
FROM admissions AS a
GROUP BY a.department
ORDER BY admissions DESC, a.department;
""", title="Department dashboard")


# Part XXI · EXPLAIN

`EXPLAIN` shows PostgreSQL's planned execution strategy.

It does **not** execute the query when used without `ANALYZE`.

The goal at this point is not to become a query planner expert.

Instead, notice that SQL written with a subquery may be transformed internally into an efficient plan.


## 31. Explain an EXISTS query


In [ ]:
display_query("""
EXPLAIN
SELECT
    p.patient_id,
    p.name
FROM patients AS p
WHERE EXISTS (
    SELECT 1
    FROM admissions AS a
    WHERE a.patient_id = p.patient_id
);
""", title="EXPLAIN EXISTS")


### Performance habits

1. Write correct logic first.
2. Make inner-query shape explicit.
3. Avoid unnecessary repeated work.
4. Prefer existence logic for existence questions.
5. Consider pre-aggregation when the same grouped result is reused.
6. Add indexes based on real query patterns, not guesses.
7. Use `EXPLAIN` when performance actually matters.


# Part XXII · Validation Habits

Before trusting a subquery, validate both the inner and outer pieces.


## Validation checklist

### Inner query

- Should it return one value or many?
- Does it unexpectedly contain NULL?
- Can it return zero rows?
- Can it return duplicates?
- Does it use the intended correlation key?

### Outer query

- What is the output grain?
- Did membership logic duplicate rows?
- Did correlation remove or repeat rows?
- Does a NULL inner result affect the predicate?
- Can a tie return more than one row?


## 32. Validate a scalar benchmark explicitly


In [ ]:
display_query("""
SELECT
    COUNT(*) AS benchmark_rows
FROM (
    SELECT AVG(age) AS average_age
    FROM patients
) AS benchmark;
""", title="Scalar benchmark should contain one row")


## 33. Check the review queue for NULL before using NOT IN


In [ ]:
display_query("""
SELECT
    COUNT(*) AS queue_rows,
    COUNT(patient_id) AS non_null_patient_ids,
    COUNT(*) - COUNT(patient_id) AS null_patient_ids
FROM review_queue;
""", title="NULL check for membership set")


# Part XXIII · Guided Practice

For each task, write the expected inner-query shape before writing SQL.

Examples:

```text
Inner shape: one scalar value
```

or:

```text
Inner shape: a set of patient IDs
```

or:

```text
Inner purpose: existence only
```


## Practice 1 · Scalar benchmark

Return patients older than the overall average age.

<details>
<summary>Hint</summary>

Inner shape: **one value**.

Run `SELECT AVG(age) FROM patients` first.

</details>


In [ ]:
practice_1 = """
SELECT
    patient_id,
    name,
    age
FROM patients
WHERE age >
    -- scalar subquery
ORDER BY age DESC;
"""

# display_query(practice_1, title="Practice 1")


## Practice 2 · Correlated count

Return one row per patient with the number of admissions.

<details>
<summary>Hint</summary>

The inner query should depend on the current outer patient's `patient_id`.

</details>


In [ ]:
practice_2 = """
SELECT
    p.patient_id,
    p.name,
    (
        SELECT
            -- count
        FROM admissions AS a
        WHERE
            -- correlation
    ) AS admission_count
FROM patients AS p
ORDER BY p.patient_id;
"""


## Practice 3 · EXISTS

Return patients who have at least one abnormal lab.

The outer result should still be one row per patient.


In [ ]:
practice_3 = """
SELECT
    p.patient_id,
    p.name
FROM patients AS p
WHERE EXISTS (
    SELECT 1
    FROM lab_results AS l
    WHERE
        -- correlation
        -- abnormal condition
)
ORDER BY p.patient_id;
"""


## Practice 4 · Distinct medication count

Return one row per patient with the number of distinct medications.

Patients with no prescriptions should receive 0.


In [ ]:
practice_4 = """
SELECT
    p.patient_id,
    p.name,
    (
        SELECT
            -- distinct count
        FROM prescriptions AS pr
        WHERE
            -- correlation
    ) AS medication_count
FROM patients AS p
ORDER BY p.patient_id;
"""


## Practice 5 · NOT EXISTS

Return patients who are not in the review queue.

Do not use `NOT IN`.


In [ ]:
practice_5 = """
SELECT
    p.patient_id,
    p.name
FROM patients AS p
WHERE NOT EXISTS (
    SELECT 1
    FROM review_queue AS q
    WHERE
        -- correlation
)
ORDER BY p.patient_id;
"""


# Part XXIV · Final Challenge

Return patients whose admission count is **above the average admission count among patients who have at least one admission**.

This requires several reasoning steps.

### Step 1

Build a table-like result:

```text
patient_id | admission_count
```

### Step 2

Calculate the average of those patient-level counts.

### Step 3

Compare each patient's admission count with that average.

### Requirement

Use:
- a derived table or CTE,
- at least one subquery,
- a clear one-row-per-patient final result.


In [ ]:
final_query = """
WITH admission_counts AS (
    SELECT
        patient_id,
        COUNT(*) AS admission_count
    FROM admissions
    GROUP BY patient_id
)
SELECT
    p.patient_id,
    p.name,
    -- admission count
FROM patients AS p
-- join or subquery logic
WHERE
    -- compare with average admission_count
ORDER BY p.patient_id;
"""

# display_query(final_query, title="Final challenge")


# Quick Concept Check

### What must a scalar subquery return?
<details><summary>Show answer</summary>One column and at most one row.</details>

### What happens when a scalar subquery returns zero rows?
<details><summary>Show answer</summary>Its value becomes NULL.</details>

### Why can IN and JOIN return different row counts?
<details><summary>Show answer</summary>IN tests membership while JOIN returns matching row combinations and may repeat outer entities.</details>

### Why is NOT IN dangerous when the inner set contains NULL?
<details><summary>Show answer</summary>NULL introduces UNKNOWN comparisons, which can cause expected rows to be filtered out.</details>

### What question does EXISTS answer?
<details><summary>Show answer</summary>Whether at least one matching inner row exists.</details>

### What makes a subquery correlated?
<details><summary>Show answer</summary>It references a value from the current outer query row.</details>

### What is a derived table?
<details><summary>Show answer</summary>A subquery in FROM that behaves like a temporary table and requires an alias.</details>

### What is the best first debugging step for a subquery?
<details><summary>Show answer</summary>Run the inner query by itself and verify its shape and values.</details>


# Module 7 Summary

Choose subquery patterns by result shape:

| Need | Pattern |
|---|---|
| one value | scalar subquery |
| set membership | `IN` |
| existence | `EXISTS` / `NOT EXISTS` |
| comparison with a set | `ANY` / `ALL` |
| row-specific calculation | correlated subquery |
| temporary table-like result | derived table / CTE |

Use this workflow:

1. Define the outer-query grain.
2. Decide the required inner-query shape.
3. Run the inner query by itself.
4. Check NULL and duplicate behavior.
5. Embed the subquery.
6. Validate the final row count and meaning.
7. Rewrite with pre-aggregation or a CTE when it improves clarity.

## Next
**Module 8: Strings and Dates in PostgreSQL**
